In [1]:
# ============================================================
# PDB + AlphaFold Protein Structure Data Collector
# ============================================================
# USER INPUT: enter the protein name when prompted.
# Output: <protein>_PDB_AlphaFold_Data.xlsx

# %% 1. Imports and setup
import sys, subprocess, io, gzip, re, time, warnings
import json, os, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP, ROUND_HALF_EVEN
from collections import defaultdict
from datetime import datetime, timezone
from functools import lru_cache
from typing import Any, Dict, List, Optional


def ensure_package(import_name, package_name=None):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_name or import_name])

for mod, pkg in [("requests", "requests"), ("pandas", "pandas"), ("openpyxl", "openpyxl"),
                 ("pdfplumber", "pdfplumber"), ("Bio", "biopython")]:
    ensure_package(mod, pkg)

import requests
import pandas as pd
import pdfplumber
import xml.etree.ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.Align import PairwiseAligner
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

In [2]:
# 2. User input and performance settings
# The user only needs to provide a protein name, gene symbol, recognized alias, or UniProt accession.
protein_name = input("Enter protein name: ").strip()
if not protein_name:
    raise ValueError("Protein name cannot be empty.")

HUMAN_TAXONOMY_ID = 9606
REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_BACKOFF = 0.8
RATE_LIMIT_DELAY = 0.03
MAX_WORKERS = min(12, max(4, (os.cpu_count() or 4) * 2))

def safe_filename(text):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text).strip())
    text = re.sub(r"_+", "_", text).strip("._")
    return text or "Protein"

OUTPUT_FILENAME = f"{safe_filename(protein_name)}_PDB_AlphaFold_Data.xlsx"

print(f"Protein input : {protein_name}")
print(f"Output file   : {OUTPUT_FILENAME}")
print(f"Worker threads: {MAX_WORKERS}")


Enter protein name:  CDK4


Protein input : CDK4
Output file   : CDK4_PDB_AlphaFold_Data.xlsx
Worker threads: 12


In [3]:
# 3. API endpoints and fast HTTP helpers
UNIPROT_SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"
UNIPROT_ENTRY_URL = "https://rest.uniprot.org/uniprotkb/{accession}.json"
UNIPROT_BROWSER_URL = "https://www.uniprot.org/uniprotkb/{accession}"

RCSB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_ENTRY_URL = "https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
RCSB_POLYMER_URL = "https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}"
RCSB_POLYMER_INSTANCE_URL = "https://data.rcsb.org/rest/v1/core/polymer_entity_instance/{pdb_id}/{asym_id}"
RCSB_NONPOLY_URL = "https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{entity_id}"
MMCIF_URL = "https://files.rcsb.org/download/{pdb_id}.cif"
VALIDATION_XML_URL = "https://files.rcsb.org/pub/pdb/validation_reports/{mid}/{pid}/{pid}_validation.xml.gz"
VALIDATION_PDF_URL = "https://files.rcsb.org/pub/pdb/validation_reports/{mid}/{pid}/{pid}_full_validation.pdf.gz"

# AlphaFold DB primary endpoint. If that endpoint is unavailable, the EMBL-EBI
# 3D-Beacons machine-readable service is used, restricted to provider=AlphaFold DB.
ALPHAFOLD_API_URL = "https://alphafold.com/api/prediction/{accession}"
ALPHAFOLD_ENTRY_URL = "https://alphafold.ebi.ac.uk/entry/{identifier}"
THREED_BEACONS_URL = "https://www.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/api/uniprot/summary/{accession}.json"

_thread_local = threading.local()

def make_session():
    s = requests.Session()
    retry = Retry(total=MAX_RETRIES, connect=MAX_RETRIES, read=MAX_RETRIES,
                  backoff_factor=RETRY_BACKOFF,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=frozenset({"GET", "POST"}),
                  raise_on_status=False)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
    s.mount("https://", adapter)
    s.headers.update({"User-Agent": "ProteinPDBDataCollector/3.0"})
    return s

def get_session():
    if not hasattr(_thread_local, "session"):
        _thread_local.session = make_session()
    return _thread_local.session

def request_json(url, params=None, timeout=REQUEST_TIMEOUT):
    time.sleep(RATE_LIMIT_DELAY)
    try:
        r = get_session().get(url, params=params, timeout=timeout)
        r.raise_for_status()
        return r.json(), None
    except (requests.RequestException, ValueError) as e:
        return None, str(e)

def request_json_exact(url, params=None, timeout=REQUEST_TIMEOUT):
    """JSON with numeric values kept as strings so source precision is preserved."""
    time.sleep(RATE_LIMIT_DELAY)
    try:
        r = get_session().get(url, params=params, timeout=timeout)
        r.raise_for_status()
        return json.loads(r.text, parse_float=str, parse_int=str), None
    except (requests.RequestException, ValueError) as e:
        return None, str(e)

def request_bytes(url, timeout=REQUEST_TIMEOUT):
    time.sleep(RATE_LIMIT_DELAY)
    try:
        r = get_session().get(url, timeout=timeout)
        r.raise_for_status()
        return r.content, None
    except requests.RequestException as e:
        return None, str(e)

def rcsb_search(query):
    time.sleep(RATE_LIMIT_DELAY)
    try:
        r = get_session().post(RCSB_SEARCH_URL, json=query, timeout=REQUEST_TIMEOUT,
                                headers={"Content-Type": "application/json"})
        r.raise_for_status()
        return r.json(), None
    except (requests.RequestException, ValueError) as e:
        return None, str(e)

print("API/session setup complete.")


API/session setup complete.


In [4]:
# 4. Protein identity resolution through UniProt

def norm_text(value):
    value = str(value or "").lower().strip()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def uniprot_query(query, size=25):
    """
    Search UniProtKB using the current REST API.

    Only valid/current return fields are requested.
    """
    params = {
        "query": query,
        "format": "json",
        "size": size,
        "fields": "accession,id,protein_name,gene_names,organism_name,reviewed"
    }
    data, err = request_json(
        UNIPROT_SEARCH_URL,
        params=params)
    return ((data or {}).get("results", []), err)

def collect_result_names(item):
    names = set()
    pdsc = item.get("proteinDescription") or {}
    rec = pdsc.get("recommendedName") or {}
    full = rec.get("fullName") or {}
    if full.get("value"): names.add(str(full["value"]))
    for x in rec.get("shortNames", []) or []:
        if isinstance(x, dict) and x.get("value"): names.add(str(x["value"]))
    for x in pdsc.get("alternativeNames", []) or []:
        if not isinstance(x, dict): continue
        full = x.get("fullName") or {}
        if full.get("value"): names.add(str(full["value"]))
        for y in x.get("shortNames", []) or []:
            if isinstance(y, dict) and y.get("value"): names.add(str(y["value"]))
    for v in (item.get("protein_name"), item.get("id")):
        if isinstance(v, str) and v.strip(): names.add(v)
    for g in item.get("genes", []) or []:
        if not isinstance(g, dict): continue
        gn = g.get("geneName") or g.get("gene_name")
        if isinstance(gn, dict) and gn.get("value"): names.add(str(gn["value"]))
        for syn in g.get("synonyms", []) or []:
            if isinstance(syn, dict) and syn.get("value"): names.add(str(syn["value"]))
    flat = item.get("gene_names")
    if isinstance(flat, str): names.update(re.split(r"[;, ]+", flat))
    return {str(x).strip() for x in names if str(x).strip()}

def score_uniprot_match(item, user_input):
    u = norm_text(user_input)
    names = {norm_text(x) for x in collect_result_names(item)}
    score = 0
    if u in names:
        score += 100
    elif any(u in n or n in u for n in names if n):
        score += 60
    if str(item.get("reviewed", "")).lower() in {"true", "yes"}:
        score += 10
    return score

def resolve_protein_identity(user_input):
    acc_like = bool(re.fullmatch(r"[A-NR-Z][0-9][A-Z0-9]{3}[0-9]", user_input.upper()))
    queries = []
    if acc_like:
        queries.append(f"accession:{user_input.upper()} AND organism_id:{HUMAN_TAXONOMY_ID}")
    queries.extend([
        f"gene_exact:{user_input} AND organism_id:{HUMAN_TAXONOMY_ID}",
        f"gene:{user_input} AND organism_id:{HUMAN_TAXONOMY_ID}",
        f"protein_name:\"{user_input}\" AND organism_id:{HUMAN_TAXONOMY_ID}",
    ])
    results, errors = {}, []
    for q in queries:
        rows, err = uniprot_query(q)
        if err: errors.append(err)
        for row in rows:
            acc = row.get("primaryAccession")
            if acc: results[acc] = row
    if not results:
        raise RuntimeError(f"Could not resolve a human UniProt entry for '{user_input}'. API errors: {errors[:2]}")
    ranked = sorted(results.values(), key=lambda x: (
        score_uniprot_match(x, user_input),
        str(x.get("reviewed", "")).lower() == "true"
    ), reverse=True)
    top_score = score_uniprot_match(ranked[0], user_input)
    tied = [x for x in ranked if score_uniprot_match(x, user_input) == top_score]
    if len(tied) > 1 and top_score < 100:
        candidates = [x.get("primaryAccession", "?") for x in tied[:10]]
        raise RuntimeError(
            f"Protein name is ambiguous; top human UniProt candidates: {candidates}. "
            "Please provide the gene symbol or UniProt accession."
        )
    return ranked[0]

IDENTITY_RESULT = resolve_protein_identity(protein_name)
UNIPROT_ACCESSION = IDENTITY_RESULT["primaryAccession"]
ORGANISM_ID = HUMAN_TAXONOMY_ID

UNIPROT_FULL, _uniprot_err = request_json(UNIPROT_ENTRY_URL.format(accession=UNIPROT_ACCESSION))
if not UNIPROT_FULL:
    raise RuntimeError(f"UniProt entry retrieval failed for {UNIPROT_ACCESSION}: {_uniprot_err}")

protein_display_name = (
    ((UNIPROT_FULL.get("proteinDescription") or {}).get("recommendedName") or {})
    .get("fullName", {}).get("value") or protein_name
)

TARGET_NAMES = set(collect_result_names(UNIPROT_FULL))
TARGET_NAMES.update({protein_name, protein_display_name})
TARGET_NAMES = {x.strip() for x in TARGET_NAMES if len(str(x).strip()) >= 3}

TARGET_GENES = set()
for g in UNIPROT_FULL.get("genes", []) or []:
    if not isinstance(g, dict): continue
    gn = g.get("geneName") or g.get("gene_name")
    if isinstance(gn, dict) and gn.get("value"): TARGET_GENES.add(str(gn["value"]).strip())
    for syn in g.get("synonyms", []) or []:
        if isinstance(syn, dict) and syn.get("value"): TARGET_GENES.add(str(syn["value"]).strip())
TARGET_GENES = {x for x in TARGET_GENES if x}

TARGET_SEQUENCE = (((UNIPROT_FULL.get("sequence") or {}).get("value")) or "").replace(" ", "").replace("\n", "").upper()
PROTEIN_NAME_COLUMN_VALUE = f"{protein_name} ({UNIPROT_ACCESSION})"

print(f"Resolved UniProt : {UNIPROT_ACCESSION}")
print(f"Official name    : {protein_display_name}")
print(f"Gene symbols     : {', '.join(sorted(TARGET_GENES)) or 'N/A'}")
print(f"Aliases collected: {len(TARGET_NAMES)}")


Resolved UniProt : P11802
Official name    : Cyclin-dependent kinase 4
Gene symbols     : CDK4
Aliases collected: 4


In [5]:
# 5. Exact/display-value helpers
# IMPORTANT:
# Values that are displayed on PDB/wwPDB pages are deliberately stored as TEXT
# in Excel.  This prevents Excel/pandas from converting 2.10 -> 2.1 or 1 -> 1.0.
# The formatting below follows the precision normally displayed for the
# corresponding PDB/wwPDB metric:
#   Resolution                 -> 2 decimal places
#   R-value Free / Work       -> 3 decimal places
#   wwPDB DCC R-free          -> 3 decimal places
#   Clashscore                 -> nearest whole number
#   Rama/sidechain/RSRZ       -> 1 decimal place
#   Average B                  -> value extracted directly from validation PDF
#
# No missing value is replaced with a made-up number; missing values are N/A.

# ROUNDING RULE (this was the cause of the wrong values):
# wwPDB/RCSB store e.g. Ramachandran = 5.65 and Sidechain = 13.85 (2 decimals) and DISPLAY them
# with 1 decimal using "round half to even" -> 5.6 and 13.8.
# The old script used ROUND_HALF_UP -> 5.7 and 13.9 (wrong).
# All arithmetic is done with Decimal on the ORIGINAL TEXT, never through float.
DISPLAY_ROUNDING = ROUND_HALF_EVEN

MISSING_MARKERS = {"", "n/a", "na", ".", "?", "none", "null", "-", "--"}


def raw_value(value):
    """Return a source value as TEXT, using N/A for missing values."""
    if value is None:
        return "N/A"
    value = str(value).strip()
    return "N/A" if value.lower() in MISSING_MARKERS else value


def decimal_from_text(value):
    if value is None or str(value).strip().lower() in MISSING_MARKERS:
        return None
    try:
        return Decimal(str(value).strip())
    except (InvalidOperation, ValueError, TypeError):
        return None


def display_decimal(value, places=None, integer=False):
    """Format a numeric source value for the value shown by the PDB/wwPDB UI.

    The returned value is ALWAYS a string, so Excel cannot change its
    representation.  Trailing zeroes are retained only when the UI metric
    convention requires fixed precision (e.g. resolution = 2.10).
    """
    d = decimal_from_text(value)
    if d is None:
        return "N/A"

    if integer:
        q = Decimal("1")
        return format(d.quantize(q, rounding=DISPLAY_ROUNDING), "f")

    if places is None:
        return format(d, "f")

    q = Decimal("1").scaleb(-places)
    return format(d.quantize(q, rounding=DISPLAY_ROUNDING), f".{places}f")


def percentage_display(value):
    """PDB validation percentages: one decimal, except zero is shown as 0."""
    d = decimal_from_text(value)
    if d is None:
        return "N/A"
    if d == 0:
        return "0"
    return display_decimal(d, places=1)


def exact_difference(a, b):
    """Calculate a-b using Decimal without binary floating-point rounding."""
    x, y = decimal_from_text(a), decimal_from_text(b)
    if x is None or y is None:
        return "N/A"
    return format(x - y, "f")

In [6]:
# ============================================================
# 5. RCSB PDB CANDIDATE DISCOVERY
# ============================================================
#
# Uses the existing HTTP/API functions from this notebook:
#     rcsb_search()
#
# Candidate discovery routes:
#     1. UniProt accession
#     2. Gene symbols
#     3. Actual polymer-entity names
#
# IMPORTANT:
# This is only candidate discovery.
#
# Strict target identity, Homo sapiens and X-ray validation
# are performed in Section 11.
# ============================================================


def _terminal(attribute, operator, value):
    """Create one RCSB Search API terminal query."""

    return {
        "type": "terminal",
        "service": "text",
        "parameters": {
            "attribute": attribute,
            "operator": operator,
            "value": value
        }
    }


def _group(logical_operator, nodes):
    """Create an RCSB Boolean query group."""

    nodes = [x for x in nodes if x]

    if not nodes:
        return None

    if len(nodes) == 1:
        return nodes[0]

    return {
        "type": "group",
        "logical_operator": logical_operator,
        "nodes": nodes
    }


# ------------------------------------------------------------
# 1. Search by UniProt accession
# ------------------------------------------------------------

def search_pdb_by_uniprot(accession):

    if not accession:
        return set()

    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute":
                    "rcsb_polymer_entity_container_identifiers."
                    "reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": accession
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": 1000
            },
            "results_content_type": [
                "experimental"
            ]
        }
    }

    data, err = rcsb_search(query)

    if err:
        print(f"  UniProt search failed: {err}")
        return set()

    return {
        str(x.get("identifier", "")).upper().strip()
        for x in (data or {}).get("result_set", [])
        if x.get("identifier")
    }


# ------------------------------------------------------------
# 2. Search by gene symbol
# ------------------------------------------------------------

def search_pdb_by_gene(gene):

    gene = str(gene or "").strip()

    if len(gene) < 2:
        return set()

    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute":
                    "rcsb_entity_source_organism."
                    "rcsb_gene_name.value",
                "operator": "exact_match",
                "value": gene
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": 1000
            },
            "results_content_type": [
                "experimental"
            ]
        }
    }

    data, err = rcsb_search(query)

    if err:
        print(
            f"  Gene search failed for {gene}: {err}"
        )
        return set()

    return {
        str(x.get("identifier", "")).upper().strip()
        for x in (data or {}).get("result_set", [])
        if x.get("identifier")
    }


# ------------------------------------------------------------
# 3. Search actual polymer-entity name
# ------------------------------------------------------------

def search_pdb_by_entity_name(name):

    name = str(name or "").strip()

    if len(norm_text(name)) < 4:
        return set()

    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute":
                    "rcsb_polymer_entity.pdbx_description",
                "operator": "contains_phrase",
                "value": name
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": 1000
            },
            "results_content_type": [
                "experimental"
            ]
        }
    }

    data, err = rcsb_search(query)

    if err:
        print(
            f"  Entity-name search failed for "
            f"'{name}': {err}"
        )
        return set()

    return {
        str(x.get("identifier", "")).upper().strip()
        for x in (data or {}).get("result_set", [])
        if x.get("identifier")
    }


# ------------------------------------------------------------
# 4. Main candidate discovery
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RCSB PDB CANDIDATE DISCOVERY")
print("=" * 60)

all_pdb_candidates = set()

# ============================================================
# ROUTE 1 — UniProt accession
# ============================================================

print(
    f"\n[1] UniProt accession: {UNIPROT_ACCESSION}"
)

uniprot_ids = search_pdb_by_uniprot(
    UNIPROT_ACCESSION
)

print(
    f"    -> {len(uniprot_ids)} candidates"
)

all_pdb_candidates.update(uniprot_ids)


# ============================================================
# ROUTE 2 — Gene symbols
# ============================================================

genes = sorted({
    str(g).strip()
    for g in TARGET_GENES
    if str(g).strip()
})

print(
    f"\n[2] Gene-symbol searches: {len(genes)}"
)

for gene in genes:

    print(f"    Searching: {gene}")

    ids = search_pdb_by_gene(gene)

    print(
        f"        -> {len(ids)} candidates"
    )

    all_pdb_candidates.update(ids)


# ============================================================
# ROUTE 3 — Polymer entity names
# ============================================================

# Only use meaningful protein names.
# Very short terms such as "ITK", "EMT", "LYK" are deliberately
# NOT sent to the free-text entity-name search because they can
# produce unrelated matches.
#
# Those short identifiers are already searched through the
# structured gene-symbol route above.

entity_names = sorted(
    {
        str(name).strip()
        for name in TARGET_NAMES
        if (
            str(name).strip()
            and len(norm_text(name)) >= 8
        )
    },
    key=lambda x: (
        -len(norm_text(x)),
        norm_text(x)
    )
)

# Limit redundant name queries.
entity_names = entity_names[:20]

print(
    f"\n[3] Polymer entity-name searches: "
    f"{len(entity_names)}"
)

for name in entity_names:

    print(
        f"    Searching: {name}"
    )

    ids = search_pdb_by_entity_name(name)

    print(
        f"        -> {len(ids)} candidates"
    )

    all_pdb_candidates.update(ids)


# ============================================================
# Final candidate list
# ============================================================

PDB_CANDIDATE_IDS = sorted(
    {
        str(pdb_id).upper().strip()
        for pdb_id in all_pdb_candidates
        if pdb_id
    }
)

print("\n" + "-" * 60)

print(
    f"Total unique RCSB PDB candidates: "
    f"{len(PDB_CANDIDATE_IDS)}"
)

print("-" * 60)

if PDB_CANDIDATE_IDS:
    print(
        "First candidates:",
        ", ".join(PDB_CANDIDATE_IDS[:20])
    )
else:
    print(
        "No RCSB candidates were discovered."
    )


RCSB PDB CANDIDATE DISCOVERY

[1] UniProt accession: P11802
    -> 15 candidates

[2] Gene-symbol searches: 1
    Searching: CDK4
        -> 7 candidates

[3] Polymer entity-name searches: 2
    Searching: Cell division protein kinase 4
        -> 5 candidates
    Searching: Cyclin-dependent kinase 4
        -> 12 candidates

------------------------------------------------------------
Total unique RCSB PDB candidates: 17
------------------------------------------------------------
First candidates: 1D9S, 1DC2, 2W96, 2W99, 2W9F, 2W9Z, 3G33, 5FWK, 5FWL, 5FWM, 5FWP, 6P8E, 6P8F, 6P8G, 6P8H, 7SJ3, 9CSK


In [7]:
# 7. RCSB entry, method, resolution, residue counts and chains

def get_method(entry):
    methods = [x.get("method", "").strip() for x in entry.get("exptl", []) if x.get("method")]
    return ", ".join(dict.fromkeys(methods)) or "N/A"

def get_resolution(entry):
    vals = (entry.get("rcsb_entry_info") or {}).get("resolution_combined") or []
    vals = vals if isinstance(vals, list) else [vals]
    for value in vals:
        if decimal_from_text(value) is not None:
            return display_decimal(value, places=2)
    return "N/A"

def _entity_asym_ids(entity):
    asym = ((entity.get("rcsb_polymer_entity_container_identifiers") or {}).get("asym_ids") or [])
    if isinstance(asym, str): asym = [asym]
    return [str(x).strip() for x in asym if str(x).strip()]

def get_residue_counts(entry, pdb_id, entity_cache=None):
    deposited_raw = (entry.get("rcsb_entry_info") or {}).get("deposited_polymer_monomer_count")
    try:
        deposited = int(deposited_raw) if deposited_raw is not None else None
    except (TypeError, ValueError):
        deposited = None
    if deposited is None:
        return None, None, None

    polymer_entity_ids = (entry.get("rcsb_entry_container_identifiers") or {}).get("polymer_entity_ids") or []
    entity_cache = entity_cache or {}
    entities=[]
    missing_ids=[]
    for eid in polymer_entity_ids:
        ent=entity_cache.get(str(eid))
        if ent is not None:
            entities.append(ent)
        else:
            missing_ids.append(eid)
    if missing_ids:
        def fetch(eid):
            return str(eid), request_json_exact(RCSB_POLYMER_URL.format(pdb_id=pdb_id.upper(), entity_id=eid))[0]
        with ThreadPoolExecutor(max_workers=min(6,max(1,len(missing_ids)))) as ex:
            for eid,ent in ex.map(fetch,missing_ids):
                if ent: entities.append(ent)

    asym_ids=[]; seen=set()
    for entity in entities:
        for asym_id in _entity_asym_ids(entity):
            if asym_id not in seen:
                seen.add(asym_id); asym_ids.append(asym_id)

    def fetch_instance(asym_id):
        return request_json_exact(RCSB_POLYMER_INSTANCE_URL.format(pdb_id=pdb_id.upper(), asym_id=asym_id))[0]
    modeled_total=0; found=False
    with ThreadPoolExecutor(max_workers=min(8,max(1,len(asym_ids)))) as ex:
        for instance in ex.map(fetch_instance,asym_ids):
            if not instance: continue
            modeled=((instance.get("rcsb_polymer_instance_info") or {}).get("modeled_residue_count"))
            if modeled is None: continue
            try:
                modeled_total += int(modeled); found=True
            except (TypeError,ValueError): pass
    modeled=modeled_total if found else None
    difference=deposited-modeled if modeled is not None else None
    return deposited,modeled,difference

def get_pdb_chains(entry, pdb_id, entity_cache=None):
    ids=(entry.get("rcsb_entry_container_identifiers") or {}).get("polymer_entity_ids") or []
    entity_cache=entity_cache or {}
    chains=[]; missing=[]
    for eid in ids:
        data=entity_cache.get(str(eid))
        if data is not None:
            asym=_entity_asym_ids(data)
            for chain in asym:
                if chain not in chains: chains.append(chain)
        else:
            missing.append(eid)
    if missing:
        def fetch(eid): return request_json_exact(RCSB_POLYMER_URL.format(pdb_id=pdb_id.upper(), entity_id=eid))[0]
        with ThreadPoolExecutor(max_workers=min(6,max(1,len(missing)))) as ex:
            for data in ex.map(fetch,missing):
                if not data: continue
                for chain in _entity_asym_ids(data):
                    if chain not in chains: chains.append(chain)
    return ", ".join(chains) if chains else "N/A"

In [8]:
# 8. Depositor R-values from mmCIF
def get_depositor_r_values(pdb_id):
    """Read depositor R-values and format them to the PDB page precision."""
    raw, err = request_bytes(MMCIF_URL.format(pdb_id=pdb_id.upper()))
    if err or raw is None:
        return None, None
    try:
        cif = MMCIF2Dict(io.StringIO(raw.decode("utf-8", errors="replace")))

        def first(key):
            v = cif.get(key)
            if isinstance(v, list):
                v = next((x for x in v if str(x).strip() not in MISSING_MARKERS), None)
            if v is None or str(v).strip().lower() in MISSING_MARKERS:
                return None
            return str(v).strip()

        rfree = first("_refine.ls_R_factor_R_free")
        rwork = first("_refine.ls_R_factor_R_work")

        # PDB entry pages display these values to three decimal places.
        return (
            display_decimal(rfree, places=3) if rfree is not None else None,
            display_decimal(rwork, places=3) if rwork is not None else None,
        )
    except Exception:
        return None, None

In [9]:
# 9. wwPDB validation XML + Average B-factor
# IMPORTANT:
# Validation metrics are NOT taken only from the RCSB entry JSON.
# The old working script fetched the official wwPDB validation XML first,
# then used the RCSB entry API as a fallback. This same logic is restored here.
#
# Metrics collected:
#   - R-free (DCC)
#   - Clashscore
#   - Ramachandran Outliers (%)
#   - Sidechain/Rotamer Outliers (%)
#   - RSRZ Outliers (%)
#   - Average B, all atoms (Å²)

AVERAGE_B_PATTERN = re.compile(
    r"Average\s*B,\s*all\s*atoms\s*\([^)]*\)\s*(-?\d+\.\d+)",
    re.I
)


def get_validation(pdb_id):
    """
    Fetch official wwPDB validation XML for one PDB entry.

    Returns:
        (validation_attributes, error)
    """
    pid = str(pdb_id).strip().lower()
    mid = pid[1:3]

    raw, err = request_bytes(
        VALIDATION_XML_URL.format(mid=mid, pid=pid)
    )

    if err or raw is None:
        return {}, err

    try:
        root = ET.fromstring(gzip.decompress(raw))
        node = root.find("Entry")

        if node is None:
            return {}, "no Entry element in validation XML"

        return dict(node.attrib), None

    except Exception as e:
        return {}, str(e)


def get_average_b(pdb_id):
    """
    Extract 'Average B, all atoms' directly from the official
    wwPDB validation PDF.
    """
    pid = str(pdb_id).strip().lower()
    mid = pid[1:3]

    raw, err = request_bytes(
        VALIDATION_PDF_URL.format(mid=mid, pid=pid)
    )

    if err or raw is None:
        return "N/A"

    try:
        with pdfplumber.open(
            io.BytesIO(gzip.decompress(raw))
        ) as pdf:

            for page in pdf.pages:
                text = page.extract_text() or ""

                match = AVERAGE_B_PATTERN.search(text)

                if match:
                    return raw_value(match.group(1))

                for line in text.splitlines():
                    if "Average B" in line and "atom" in line.lower():
                        nums = re.findall(
                            r"-?\d+(?:\.\d+)?",
                            line
                        )
                        if nums:
                            return raw_value(nums[-1])

    except Exception:
        pass

    return "N/A"


In [10]:
# 9a. RCSB API validation values (exact TEXT fallback)
# This is intentionally kept as a SECOND source.
# Primary source = wwPDB validation XML.
# Fallback      = RCSB entry API.
#
# parse_float=str / parse_int=str prevents loss of source precision.

def request_json_exact(url, params=None, timeout=REQUEST_TIMEOUT):
    """Fetch JSON while preserving server numeric values as TEXT."""
    time.sleep(RATE_LIMIT_DELAY)

    try:
        r = get_session().get(
            url,
            params=params,
            timeout=timeout
        )
        r.raise_for_status()

        return json.loads(
            r.text,
            parse_float=str,
            parse_int=str
        ), None

    except (requests.RequestException, ValueError) as e:
        return None, str(e)


def _first_dict(x):
    if isinstance(x, list):
        x = x[0] if x else {}

    return x if isinstance(x, dict) else {}


def get_api_validation_raw(pdb_id):
    """
    Retrieve raw validation metrics from the RCSB entry API.

    Used only as a fallback when the wwPDB validation XML does not
    contain a particular metric.
    """
    data, err = request_json_exact(
        RCSB_ENTRY_URL.format(
            pdb_id=str(pdb_id).upper()
        )
    )

    out = {}

    if err or not data:
        return out

    summ = _first_dict(
        data.get("pdbx_vrpt_summary")
    )

    diff = _first_dict(
        data.get("pdbx_vrpt_summary_diffrn")
    )

    out["clashscore"] = summ.get("clashscore")
    out["percent-rama-outliers"] = (
        summ.get("percent_ramachandran_outliers")
    )
    out["percent-rota-outliers"] = (
        summ.get("percent_rotamer_outliers")
    )
    out["percent-RSRZ-outliers"] = (
        summ.get("percent_rsrzoutliers")
    )

    out["DCC_Rfree"] = (
        diff.get("dcc_rfree")
        or summ.get("dcc_rfree")
    )

    return out


In [11]:
# 10. Ligand and binding-activity extraction
COMMON_NON_LIGAND_COMPONENTS = {
    "HOH","H2O","WAT","NA","CL","K","MG","CA","ZN","MN","CD","CO","NI","CU","FE","FE2",
    "BR","IOD","CS","LI","SR","BA","RB","AL","AG","AU","HG","SO4","PO4","GOL","EDO","PEG",
    "PG4","MPD","ACT","TRS","IPA","DMS","BME","MES","HEPES","CIT","FMT","ACY","1PE","PGE","P6G",
    "EPE","IMD","NH4","OXL","UNX","UNL","DTT","TAM","BOG","CO3","NO3","AZI"
}


def get_ligand_info(entry, pdb_id):
    ids = (entry.get("rcsb_entry_container_identifiers") or {}).get("non_polymer_entity_ids") or []
    raw = []
    for entity_id in ids:
        data, err = request_json(RCSB_NONPOLY_URL.format(pdb_id=pdb_id, entity_id=entity_id))
        if err or not data:
            continue
        info = data.get("pdbx_entity_nonpoly") or {}
        comp = str(info.get("comp_id", "")).strip()
        if comp:
            raw.append({"comp_id": comp, "name": info.get("name") or comp})
    relevant = [x for x in raw if x["comp_id"].upper() not in COMMON_NON_LIGAND_COMPONENTS]
    return relevant, raw


def get_ligand_activity(entry, ligand_ids):
    affinities = entry.get("rcsb_binding_affinity") or []
    if not affinities or not ligand_ids:
        return "N/A"
    grouped = defaultdict(lambda: defaultdict(list))
    for a in affinities:
        comp = a.get("comp_id", "")
        if comp in ligand_ids:
            grouped[comp][a.get("type", "?")].append((a.get("value"), a.get("unit", "")))
    parts = []
    for comp, types in grouped.items():
        assay_parts = []
        for assay, vals in types.items():
            nums = [(v, decimal_from_text(v)) for v, _ in vals if v is not None and decimal_from_text(v) is not None]
            unit = vals[0][1] if vals else ""
            if len(nums) == 1:
                assay_parts.append(f"{assay}: {nums[0][0]} ({unit}) from {len(vals)} assay(s)")
            elif nums:
                mn = min(nums, key=lambda x: x[1])[0]
                mx = max(nums, key=lambda x: x[1])[0]
                assay_parts.append(f"{assay}: min: {mn}, max: {mx} ({unit}) from {len(vals)} assay(s)")
        if assay_parts:
            parts.append(f"[{comp}] " + "; ".join(assay_parts))
    return " | ".join(parts) if parts else "N/A"


In [12]:
import requests

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "PDB-AlphaFold-Automation/1.0",
    "Accept": "application/json"
})

In [13]:
# 11. Strict target-entity validation

def _first_dict(x):
    if isinstance(x, list): x = x[0] if x else {}
    return x if isinstance(x, dict) else {}

def _flatten_strings(obj):
    out=[]
    if isinstance(obj,str): return [obj]
    if isinstance(obj,dict):
        for v in obj.values(): out.extend(_flatten_strings(v))
    elif isinstance(obj,list):
        for v in obj: out.extend(_flatten_strings(v))
    return out

def entity_reference_accessions(entity):
    refs=((entity.get("rcsb_polymer_entity_container_identifiers") or {}).get("reference_sequence_identifiers") or [])
    if isinstance(refs,dict): refs=[refs]
    return [
        (str(r.get("database_name","")).upper(), str(r.get("database_accession","")).strip())
        for r in refs if isinstance(r,dict) and r.get("database_accession")
    ]

def entity_human(entity):
    sources=entity.get("rcsb_entity_source_organism") or []
    if isinstance(sources,dict): sources=[sources]
    for src in sources:
        if not isinstance(src,dict): continue
        name=str(src.get("ncbi_scientific_name","")).strip().lower()
        tax=str(src.get("ncbi_taxonomy_id",src.get("taxonomy_id",""))).strip()
        if name=="homo sapiens" or tax==str(HUMAN_TAXONOMY_ID): return True
    txt=" ".join(_flatten_strings(sources)).lower()
    return "homo sapiens" in txt or re.search(r"\b9606\b",txt) is not None

def entity_texts(entity):
    texts=[]
    for key in ("pdbx_description","rcsb_macromolecular_names_combined","rcsb_polymer_name_combined"):
        texts.extend(_flatten_strings(entity.get(key)))
    return {norm_text(x) for x in texts if isinstance(x,str) and x.strip()}

def entity_gene_names(entity):
    out=set()
    sources=entity.get("rcsb_entity_source_organism") or []
    if isinstance(sources,dict): sources=[sources]
    for src in sources:
        if not isinstance(src,dict): continue
        g=src.get("rcsb_gene_name")
        if isinstance(g,dict) and g.get("value"): out.add(norm_text(g["value"]))
        elif isinstance(g,str): out.add(norm_text(g))
    return {x for x in out if x}

def entity_sequence(entity):
    seq=((entity.get("entity_poly") or {}).get("pdbx_seq_one_letter_code_can") or "")
    if not isinstance(seq,str): return ""
    return re.sub(r"[^A-Za-z]", "", seq).upper()

def sequence_identity_to_target(entity):
    seq=entity_sequence(entity)
    if not TARGET_SEQUENCE or not seq or len(seq) < 20: return None
    try:
        aligner=PairwiseAligner()
        aligner.mode="local"
        aligner.match_score=2
        aligner.mismatch_score=-1
        aligner.open_gap_score=-3
        aligner.extend_gap_score=-0.5
        aln=aligner.align(TARGET_SEQUENCE,seq)[0]
        blocks=aln.aligned
        aligned_len=sum(min(b1[1]-b1[0],b2[1]-b2[0]) for b1,b2 in zip(blocks[0],blocks[1]))
        if aligned_len < 20: return None
        # Reconstruct the aligned residue pairs from the alignment blocks.
        matches=0
        for (a0,a1),(b0,b1) in zip(blocks[0],blocks[1]):
            for i in range(a1-a0):
                if TARGET_SEQUENCE[a0+i] == seq[b0+i]: matches += 1
        identity=matches/aligned_len
        entity_coverage=aligned_len/len(seq)
        return identity,entity_coverage,aligned_len
    except Exception:
        return None

def entity_is_target(entity):
    polymer_type=str(((entity.get("entity_poly") or {}).get("rcsb_entity_polymer_type") or "")).lower()
    if polymer_type and polymer_type != "protein": return False,"not_protein"
    if not entity_human(entity): return False,"nonhuman_target"
    refs=entity_reference_accessions(entity)
    if any(db in {"UNP","UNIPROT","UNIPROTKB"} and acc.upper()==UNIPROT_ACCESSION.upper() for db,acc in refs):
        return True,"uniprot_accession_match"
    target_names={norm_text(x) for x in TARGET_NAMES}
    target_genes={norm_text(x) for x in TARGET_GENES}
    entity_names=entity_texts(entity)
    entity_genes=entity_gene_names(entity)
    if entity_names & target_names or entity_genes & target_genes:
        return True,"entity_name_or_gene_match"
    # Domain/fragment constructs often prepend text such as "SH3 domain of"
    # to the target name. A name-only phrase match is accepted only when a
    # sequence-level check also supports the target identity. This prevents
    # false positives such as a CDK4 inhibitor whose entity name contains CDK4.
    for ename in entity_names:
        if any(len(alias) >= 4 and (f" {alias} " in f" {ename} " or f" {ename} ".endswith(f" {alias}")) for alias in target_names):
            sim=sequence_identity_to_target(entity)
            if sim and sim[0] >= 0.85 and sim[2] >= 30:
                return True,"entity_name_plus_sequence_match"
    return False,"target_mismatch"

def validate_pdb_candidate(pid):
    entry,err=request_json_exact(RCSB_ENTRY_URL.format(pdb_id=pid))
    if err or not entry: return None,f"entry_error: {err}"
    methods=[str(x.get("method","")).strip().upper() for x in entry.get("exptl",[]) if isinstance(x,dict)]
    if "X-RAY DIFFRACTION" not in methods: return None,"non_xray"
    ids=(entry.get("rcsb_entry_container_identifiers") or {}).get("polymer_entity_ids") or []
    if not ids: return None,"no_polymer_entities"
    def fetch(eid):
        return str(eid),request_json_exact(RCSB_POLYMER_URL.format(pdb_id=pid,entity_id=eid))[0]
    entity_cache={}
    with ThreadPoolExecutor(max_workers=min(6,max(1,len(ids)))) as ex:
        for eid,ent in ex.map(fetch,ids):
            if ent: entity_cache[eid]=ent
    matches=[]
    for eid,ent in entity_cache.items():
        ok,reason=entity_is_target(ent)
        if ok: matches.append((eid,ent,reason))
    if not matches: return None,"target_entity_not_found"
    return {"pdb_id":pid,"entry":entry,"entity_cache":entity_cache,"target_entities":matches},None

validated=[]; validation_failures=defaultdict(int)
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures={executor.submit(validate_pdb_candidate,pid):pid for pid in PDB_CANDIDATE_IDS}
    for fut in as_completed(futures):
        pid=futures[fut]
        try:
            result,err=fut.result()
            if result: validated.append(result)
            else: validation_failures[err or "unknown"]+=1
        except Exception as e:
            validation_failures[f"exception: {e}"]+=1

validated.sort(key=lambda x:x["pdb_id"])
pdb_id_records=validated
print(f"Qualifying human X-ray target entries: {len(pdb_id_records)}")
if validation_failures:
    print("Candidate exclusions:",dict(validation_failures))
if not pdb_id_records:
    print(f"No crystal structures are available for {protein_name}.")


Qualifying human X-ray target entries: 11
Candidate exclusions: {'non_xray': 6}


In [14]:
# 12. Column definitions (Notes / Errors removed)
FINAL_COLUMNS = [
    "Protein Name", "PDB ID", "PDB Link", "Method", "Resolution (Å)",
    "Deposited Residue Count", "Modeled Residue Count", "Residue Count Difference",
    "R-Value Free", "R-Value Work", "R-Value Difference", "R-free",
    "Clashscore", "Ramachandran Outliers (%)", "Sidechain Outliers (%)", "RSRZ Outliers (%)",
    "Average B, all atoms (Å²)", "Ligand ID", "Ligand Name", "Structure Type", "PDB Chains", "Ligand Activity",
    "All HET Components (raw)"
]

print(f"PDB_Data columns: {len(FINAL_COLUMNS)}")

PDB_Data columns: 23


In [15]:
# 13. Main PDB processing -- parallelized
# Validation metrics use the same working logic as the old script:
#   1. Official wwPDB validation XML
#   2. RCSB API fallback if XML lacks a metric
# This prevents the validation columns from becoming N/A simply because
# validation fields are absent from the already-downloaded entry JSON.

VALIDATION_MAP = {
    "DCC_Rfree": "R-free",
    "clashscore": "Clashscore",
    "percent-rama-outliers": "Ramachandran Outliers (%)",
    "percent-rota-outliers": "Sidechain Outliers (%)",
    "percent-RSRZ-outliers": "RSRZ Outliers (%)",
}


def choose_validation_value(xml_data, api_data, key):
    """
    Prefer the official wwPDB validation XML value.
    If it is missing, use the RCSB API value.
    """
    value = xml_data.get(key)

    if raw_value(value) != "N/A":
        return value

    return api_data.get(key)


def format_validation_metric(key, value):
    """Apply the same display precision used by the old working script."""
    if key == "DCC_Rfree":
        return display_decimal(value, places=3)

    if key == "clashscore":
        return display_decimal(value, integer=True)

    return percentage_display(value)


def process_validated_pdb(rec):
    pid = rec["pdb_id"]
    entry = rec["entry"]
    entity_cache = rec["entity_cache"]

    row = {
        c: "N/A"
        for c in FINAL_COLUMNS
    }

    row["Protein Name"] = PROTEIN_NAME_COLUMN_VALUE
    row["PDB ID"] = pid
    row["PDB Link"] = pid

    try:
        # ---------------------------------------------------------
        # General PDB metadata
        # ---------------------------------------------------------
        row["Method"] = get_method(entry)
        row["Resolution (Å)"] = get_resolution(entry)

        dep, mod, diff = get_residue_counts(
            entry,
            pid,
            entity_cache
        )

        row["Deposited Residue Count"] = (
            dep if dep is not None else "N/A"
        )

        row["Modeled Residue Count"] = (
            mod if mod is not None else "N/A"
        )

        row["Residue Count Difference"] = (
            diff if diff is not None else "N/A"
        )

        row["PDB Chains"] = get_pdb_chains(
            entry,
            pid,
            entity_cache
        )

        # ---------------------------------------------------------
        # Depositor R-values from mmCIF
        # ---------------------------------------------------------
        rfree, rwork = get_depositor_r_values(pid)

        row["R-Value Free"] = raw_value(rfree)
        row["R-Value Work"] = raw_value(rwork)

        row["R-Value Difference"] = exact_difference(
            row["R-Value Free"],
            row["R-Value Work"]
        )

        # ---------------------------------------------------------
        # IMPORTANT: wwPDB validation metrics
        #
        # PRIMARY  -> validation XML
        # FALLBACK -> RCSB entry API
        # ---------------------------------------------------------
        validation, validation_err = get_validation(pid)
        api_validation = get_api_validation_raw(pid)

        for key, column in VALIDATION_MAP.items():

            source_value = choose_validation_value(
                validation,
                api_validation,
                key
            )

            row[column] = format_validation_metric(
                key,
                source_value
            )

        # Average B is taken from the official validation PDF.
        row["Average B, all atoms (Å²)"] = raw_value(
            get_average_b(pid)
        )

        # ---------------------------------------------------------
        # Ligand information
        # ---------------------------------------------------------
        ligands, raw = get_ligand_info(
            entry,
            pid
        )

        row["All HET Components (raw)"] = (
            ", ".join(
                x["comp_id"]
                for x in raw
            )
            if raw
            else "N/A"
        )

        if ligands:

            row["Ligand ID"] = ", ".join(
                x["comp_id"]
                for x in ligands
            )

            row["Ligand Name"] = ", ".join(
                x["name"]
                for x in ligands
            )

            row["Structure Type"] = "Complex"

            row["Ligand Activity"] = (
                get_ligand_activity(
                    entry,
                    {
                        x["comp_id"]
                        for x in ligands
                    }
                )
            )

        else:
            row["Structure Type"] = "Apo"

        # ---------------------------------------------------------
        # Force display-sensitive fields to TEXT.
        # This preserves N/A and trailing zeroes in Excel.
        # ---------------------------------------------------------
        for col in (
            "Resolution (Å)",
            "R-Value Free",
            "R-Value Work",
            "R-Value Difference",
            "R-free",
            "Clashscore",
            "Ramachandran Outliers (%)",
            "Sidechain Outliers (%)",
            "RSRZ Outliers (%)",
            "Average B, all atoms (Å²)"
        ):
            row[col] = raw_value(
                row.get(col)
            )

        return row, None

    except Exception as e:
        return row, str(e)


rows = []
failed = []

if pdb_id_records:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                process_validated_pdb,
                rec
            ): rec["pdb_id"]
            for rec in pdb_id_records
        }

        for future in as_completed(futures):

            pid = futures[future]

            try:
                row, err = future.result()

                rows.append(row)

                if err:
                    failed.append(
                        (pid, err)
                    )

            except Exception as e:
                failed.append(
                    (pid, str(e))
                )

    rows.sort(
        key=lambda x: str(
            x.get("PDB ID", "")
        )
    )

df = pd.DataFrame(
    rows,
    columns=FINAL_COLUMNS
)

print(
    f"Finished: {len(df)} qualifying PDB rows | "
    f"Failed metadata rows: {len(failed)}"
)

if failed:
    print(
        "Failed rows:",
        failed[:20]
    )


Finished: 11 qualifying PDB rows | Failed metadata rows: 0


In [16]:
# 14. AlphaFold collection -- ALWAYS executed independently of PDB result

def _af_value(obj,*keys):
    for key in keys:
        if isinstance(obj,dict) and obj.get(key) is not None: return obj.get(key)
    return None

def get_alphafold_direct(accession):
    data,err=request_json(ALPHAFOLD_API_URL.format(accession=accession))
    if err or data is None: return []
    if isinstance(data,list): items=data
    elif isinstance(data,dict): items=data.get("structures",data.get("predictions",data.get("data",[])))
    else: items=[]
    if isinstance(items,dict): items=[items]
    rows=[]
    for item in items or []:
        if not isinstance(item,dict): continue
        identifier=str(_af_value(item,"modelEntityId","model_identifier","entryId","identifier","modelIdentifier") or "").strip()
        uni=str(_af_value(item,"uniprotAccession","uniprot_accession","accession") or "").strip()
        if not identifier and uni: identifier=f"AF-{uni}-F1"
        if not identifier or (uni and uni.upper()!=accession.upper()): continue
        start=_af_value(item,"sequenceStart","uniprot_start","start"); end=_af_value(item,"sequenceEnd","uniprot_end","end")
        positions=f"{int(start)}-{int(end)}" if str(start).isdigit() and str(end).isdigit() else "N/A"
        rows.append({"Protein Name":PROTEIN_NAME_COLUMN_VALUE,"Identifier":identifier,
                     "Chain":str(_af_value(item,"chain","chainId","chain_id") or "N/A"),
                     "Positions":positions,
                     "Oligomeric State":str(_af_value(item,"oligomericState","oligomeric_state") or "N/A")})
    return rows

def get_alphafold_db_summary(accession):
    data,err=request_json(THREED_BEACONS_URL.format(accession=accession))
    if err or data is None: return []
    out=[]
    for structure in data.get("structures",[]) if isinstance(data.get("structures",[]),list) else []:
        if not isinstance(structure,dict): continue
        summary=structure.get("summary",{})
        if not isinstance(summary,dict): continue
        if str(summary.get("provider","")).strip().lower()=="alphafold db": out.append(summary)
    return out

def extract_alphafold_rows(accession):
    direct=get_alphafold_direct(accession)
    if direct:
        seen=set(); rows=[]
        for row in direct:
            key=row["Identifier"].upper()
            if key not in seen: seen.add(key); rows.append(row)
        return sorted(rows,key=lambda x:x["Identifier"].upper())
    rows=[]; seen=set()
    for summary in get_alphafold_db_summary(accession):
        identifier=str(summary.get("model_identifier","")).strip()
        if not identifier or identifier.upper() in seen: continue
        seen.add(identifier.upper())
        chains=[]
        for entity in summary.get("entities",[]) or []:
            if not isinstance(entity,dict): continue
            ids=entity.get("chain_ids",[]); ids=[ids] if isinstance(ids,str) else ids
            for chain in ids if isinstance(ids,list) else []:
                chain=str(chain).strip()
                if chain and chain not in chains and chain not in {".","?"}: chains.append(chain)
        start,end=summary.get("uniprot_start"),summary.get("uniprot_end")
        positions=f"{int(start)}-{int(end)}" if start is not None and end is not None else "N/A"
        rows.append({"Protein Name":PROTEIN_NAME_COLUMN_VALUE,"Identifier":identifier,
                     "Chain":", ".join(chains) if chains else "N/A","Positions":positions,
                     "Oligomeric State":str(summary.get("oligomeric_state") or "N/A")})
    return sorted(rows,key=lambda x:x["Identifier"].upper())

alphafold_rows=extract_alphafold_rows(UNIPROT_ACCESSION)
print(f"AlphaFold rows: {len(alphafold_rows)}")


AlphaFold rows: 1


In [17]:
# 15. Excel workbook creation
HEADER_FONT = Font(bold=True, color="FFFFFF")
HEADER_FILL = PatternFill(start_color="305496", end_color="305496", fill_type="solid")
LINK_FONT = Font(color="0563C1", underline="single")
THIN = Side(style="thin", color="B7B7B7")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def style_header(ws, columns):
    for j, col in enumerate(columns, 1):
        c = ws.cell(1, j, col); c.font = HEADER_FONT; c.fill = HEADER_FILL
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True); c.border = BORDER
    ws.row_dimensions[1].height = 28


def autofit(ws, max_width=55):
    for col in range(1, ws.max_column + 1):
        vals = [str(ws.cell(r, col).value or "") for r in range(1, ws.max_row + 1)]
        width = min(max(max(map(len, vals), default=10) + 2, 10), max_width)
        ws.column_dimensions[get_column_letter(col)].width = width

wb = Workbook()
ws = wb.active; ws.title = "PDB_Data"
style_header(ws, FINAL_COLUMNS)

DISPLAY_TEXT_COLUMNS = {
    "Resolution (Å)", "R-Value Free", "R-Value Work", "R-Value Difference",
    "R-free", "Clashscore", "Ramachandran Outliers (%)",
    "Sidechain Outliers (%)", "RSRZ Outliers (%)",
    "Average B, all atoms (Å²)"
}

for record in df.to_dict("records"):
    ws.append([record[c] for c in FINAL_COLUMNS])

for row in ws.iter_rows():
    for cell in row:
        cell.border = BORDER
        cell.alignment = Alignment(vertical="top", wrap_text=True)

# Excel number format '@' makes these cells text even when the value looks numeric.
for col_name in DISPLAY_TEXT_COLUMNS:
    col_idx = FINAL_COLUMNS.index(col_name) + 1
    for r in range(2, ws.max_row + 1):
        ws.cell(r, col_idx).number_format = "@"

for r in range(2, ws.max_row + 1):
    c = ws.cell(r, FINAL_COLUMNS.index("PDB Link") + 1)
    if c.value:
        pid = str(c.value); c.hyperlink = f"https://www.rcsb.org/structure/{pid}"; c.font = LINK_FONT

ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions; ws.sheet_view.showGridLines = False
autofit(ws)

# AlphaFold sheet

af = wb.create_sheet("AlphaFold_Data")
af_cols = ["Protein Name", "Identifier", "Chain", "Positions", "Oligomeric State"]
style_header(af, af_cols)

if alphafold_rows:
    for record in alphafold_rows:
        af.append([record.get(c, "N/A") for c in af_cols])

    identifier_col = af_cols.index("Identifier") + 1
    for r in range(2, af.max_row + 1):
        cell = af.cell(r, identifier_col)
        identifier = str(cell.value or "").strip()
        if identifier and identifier != "N/A":
            cell.hyperlink = ALPHAFOLD_ENTRY_URL.format(identifier=identifier)
            cell.font = LINK_FONT
else:
    af.append([PROTEIN_NAME_COLUMN_VALUE, "N/A", "N/A", "N/A", "N/A"])

for row in af.iter_rows():
    for cell in row:
        cell.border = BORDER
        cell.alignment = Alignment(vertical="top", wrap_text=True)

af.freeze_panes = "A2"
af.auto_filter.ref = af.dimensions
af.sheet_view.showGridLines = False
autofit(af, 45)

info=wb.create_sheet("Run_Info")
info_headers=["Field","Value"]
style_header(info,info_headers)
info_rows=[
    ("User Input",protein_name),
    ("Resolved UniProt",UNIPROT_ACCESSION),
    ("Resolved Protein Name",protein_display_name),
    ("PDB Candidates Discovered",len(PDB_CANDIDATE_IDS)),
    ("Qualifying X-ray PDB Structures",len(df)),
    ("AlphaFold Models",len(alphafold_rows)),
    ("PDB Search Status","Success"),
    ("PDB Result","Qualifying structures found" if len(df) else f"No crystal structures are available for {protein_name}."),
    ("AlphaFold Search Status","Success"),
    ("Output Generated",datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
]
for r in info_rows: info.append(list(r))
for row in info.iter_rows():
    for cell in row:
        cell.border=BORDER; cell.alignment=Alignment(vertical="top",wrap_text=True)
autofit(info,60)

wb.save(OUTPUT_FILENAME)


In [18]:
# 16. Final summary
print("\n" + "=" * 72)
print("WORKBOOK CREATED")
print("=" * 72)
print(f"File       : {OUTPUT_FILENAME}")
print(f"UniProt    : {UNIPROT_ACCESSION}")
print(f"PDB rows   : {len(df)}")
print(f"AlphaFold  : {len(alphafold_rows)}")
print(f"Failed rows: {len(failed)}")
if not len(df):
    print(f"PDB result : No crystal structures are available for {protein_name}.")
print("=" * 72)



WORKBOOK CREATED
File       : CDK4_PDB_AlphaFold_Data.xlsx
UniProt    : P11802
PDB rows   : 11
AlphaFold  : 1
Failed rows: 0


In [19]:
# ================================================================
# SECTION 17 : PDB QUALITY SCORING  (exact-value version)
# ================================================================
# What this cell guarantees
#   1. Every source value shown in PDB_Scoring / Ranges is EXACTLY the value
#      in PDB_Data (1.70 stays 1.70, 0.020 stays 0.020, 2.0 stays 2.0).
#      Values are written as real numbers with an Excel number format that
#      reproduces the original number of decimals, so they stay sortable.
#   2. Missing values are written as "N/A" (never a blank cell).
#   3. Min / Max / Median are calculated with Decimal arithmetic on the
#      ORIGINAL text. The median is shown with the same decimals as the
#      column's data (Resolution 2.065 -> 2.07), so it never shows extra digits.
#   4. The range sheet is called "Ranges".
# ================================================================

import os
from copy import copy
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP

import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment

RANGES_SHEET = "Ranges"

# ---------------------------------------------------------------
# HYPOTHESES: change the weights here (each set must add up to 1.00)
# ---------------------------------------------------------------
HYPOTHESES = {
    "PDB_Scoring_H1": {"Resolution_Score": 0.25, "Rdiff_Score": 0.25, "Residue_Score": 0.10,
                       "Validation_Quality": 0.20, "Bfactor_Score": 0.20},
    "PDB_Scoring_H2": {"Resolution_Score": 0.20, "Rdiff_Score": 0.20, "Residue_Score": 0.10,
                       "Validation_Quality": 0.25, "Bfactor_Score": 0.25},
    "PDB_Scoring_H3": {"Resolution_Score": 0.20, "Rdiff_Score": 0.20, "Residue_Score": 0.20,
                       "Validation_Quality": 0.20, "Bfactor_Score": 0.20},
}
for _name, _w in HYPOTHESES.items():
    if abs(sum(_w.values()) - 1.0) > 1e-9:
        raise ValueError(f"Weights for {_name} add up to {sum(_w.values())}, not 1.00")

# removed before re-creating (old single-sheet name included)
OLD_SHEET_NAMES = ["Scoring_Ranges", "Ranges", "PDB_Scoring"] + list(HYPOTHESES)

SCORING_COLUMNS = [
    "Resolution (Å)", "Residue Count Difference", "R-Value Difference",
    "Clashscore", "Ramachandran Outliers (%)", "Sidechain Outliers (%)",
    "RSRZ Outliers (%)", "Average B, all atoms (Å²)"
]
NA_TEXT = "N/A"


# ---------------------------------------------------------------
# Exact-number helpers
# ---------------------------------------------------------------
def to_dec(value):
    """Source value -> Decimal (exact), or None if missing / not numeric."""
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    text = str(value).strip()
    if text.lower() in {"", "n/a", "na", "nan", "none", "null", ".", "?", "-", "--"}:
        return None
    try:
        d = Decimal(text)
    except (InvalidOperation, ValueError):
        return None
    return d if d.is_finite() else None


def decimals_of(d):
    """Number of decimal places of a Decimal (2.10 -> 2, 37 -> 0, 0.0435 -> 4)."""
    return max(0, -d.as_tuple().exponent)


def excel_format(places):
    return "0" if places == 0 else "0." + "0" * places


# Median is DISPLAYED with the same number of decimals as the column's data
# (Resolution -> 2 decimals, R-Value Difference -> 3, ...), rounded half-up.
# Optional per-column override, e.g. {"Clashscore": 1} to show 3.5 instead of 4.
MEDIAN_PLACES_OVERRIDE = {}

def exact_stats(decimals, parameter=None):
    """Exact min / max, exact median (used for calculations) and the median to display."""
    vals = sorted(decimals)
    n = len(vals)
    if n == 0:
        return None
    mn, mx = vals[0], vals[-1]
    if n % 2:
        med = vals[n // 2]
    else:
        med = (vals[n // 2 - 1] + vals[n // 2]) / 2          # exact, no float
    column_places = max(decimals_of(v) for v in vals)
    places = MEDIAN_PLACES_OVERRIDE.get(parameter, column_places)
    med_shown = med.quantize(Decimal(1).scaleb(-places), rounding=ROUND_HALF_UP)
    return {"min": mn, "max": mx, "median": med, "median_shown": med_shown,
            "min_places": decimals_of(mn), "max_places": decimals_of(mx),
            "median_places": places}


def put_number(cell, d, places=None):
    """Write a Decimal as a real Excel number, formatted with the original decimals."""
    places = decimals_of(d) if places is None else places
    cell.value = int(d) if places == 0 and d == d.to_integral_value() else float(d)
    cell.number_format = excel_format(places)


# ---------------------------------------------------------------
# Numeric copies used ONLY for calculation (display text is kept untouched)
# ---------------------------------------------------------------
score_df = df.copy()
for col in SCORING_COLUMNS:
    score_df[col + "__dec"] = score_df[col].map(to_dec)
    score_df[col + "__num"] = score_df[col + "__dec"].map(
        lambda d: float(d) if d is not None else np.nan)

NUM = lambda col: score_df[col + "__num"]


# ---------------------------------------------------------------
# RANGES TABLE  (exact min / max / median)
# ---------------------------------------------------------------
range_records = []
for col in SCORING_COLUMNS:
    decs = [d for d in score_df[col + "__dec"] if d is not None]
    range_records.append({"Parameter": col, "stats": exact_stats(decs, col)})


# ---------------------------------------------------------------
# NORMALIZATION / SCORES
# ---------------------------------------------------------------
def normalize_lower_better(series):
    valid = series.dropna()
    if len(valid) == 0:
        return pd.Series(np.nan, index=series.index)
    mn, mx = valid.min(), valid.max()
    if mn == mx:
        return pd.Series(1.0, index=series.index)
    return (mx - series) / (mx - mn)

score_df["Resolution_Score"] = normalize_lower_better(NUM("Resolution (Å)"))
score_df["Residue_Score"]    = normalize_lower_better(NUM("Residue Count Difference"))
score_df["Rdiff_Score"]      = normalize_lower_better(NUM("R-Value Difference"))
score_df["Bfactor_Score"]    = normalize_lower_better(NUM("Average B, all atoms (Å²)"))

# ---------------------------------------------------------------
# VALIDATION QUALITY  (exact Decimal comparison against exact medians)
# ---------------------------------------------------------------
VALIDATION_COLUMNS = ["Clashscore", "Ramachandran Outliers (%)",
                      "Sidechain Outliers (%)", "RSRZ Outliers (%)"]
validation_medians = {}
for rec in range_records:
    if rec["Parameter"] in VALIDATION_COLUMNS and rec["stats"]:
        validation_medians[rec["Parameter"]] = rec["stats"]["median"]   # exact median, not the rounded display one

def validation_quality(row):
    results = []
    for col in VALIDATION_COLUMNS:
        d = row[col + "__dec"]
        if d is not None and col in validation_medians:
            results.append(d <= validation_medians[col])
    if not results:
        return pd.Series(["Unknown", np.nan])
    ratio = sum(results) / len(results)
    return pd.Series(["Good" if ratio >= 0.75 else "Poor", 1 if ratio >= 0.75 else 0])

if score_df.empty:
    score_df["Validation_Status"] = pd.Series(dtype=object)
    score_df["Validation_Quality"] = pd.Series(dtype=float)
else:
    score_df[["Validation_Status", "Validation_Quality"]] = score_df.apply(validation_quality, axis=1)

# ---------------------------------------------------------------
# FINAL SCORE + SORT + RANK  (one ranked table per hypothesis)
# ---------------------------------------------------------------
SCORE_COLS = ["Resolution_Score", "Residue_Score", "Rdiff_Score", "Bfactor_Score", "Final_Score"]
score_df["_res"] = NUM("Resolution (Å)")
score_df["_rd"]  = NUM("R-Value Difference")
score_df["_b"]   = NUM("Average B, all atoms (Å²)")

def rank_with_weights(weights):
    d = score_df.copy()
    d["Final_Score"] = sum(w * d[comp] for comp, w in weights.items())
    d[SCORE_COLS] = d[SCORE_COLS].round(2)
    d = d.sort_values(by=["Final_Score", "_res", "_rd", "_b"],
                      ascending=[False, True, True, True]).reset_index(drop=True)
    d["Rank"] = range(1, len(d) + 1)
    return d

OUTPUT_COLUMNS = [
    "Protein Name", "PDB ID", "PDB Link", "Resolution (Å)", "Resolution_Score",
    "R-Value Difference", "Rdiff_Score", "Residue Count Difference", "Residue_Score",
    "Clashscore", "Ramachandran Outliers (%)", "Sidechain Outliers (%)", "RSRZ Outliers (%)",
    "Validation_Status", "Validation_Quality", "Average B, all atoms (Å²)",
    "Bfactor_Score", "Final_Score", "Rank"
]
scoring_outputs = {name: rank_with_weights(w)[OUTPUT_COLUMNS].copy()   # source columns keep ORIGINAL text
                   for name, w in HYPOTHESES.items()}

# ================================================================
# WRITE TO WORKBOOK
# ================================================================
if not os.path.exists(OUTPUT_FILENAME):
    raise FileNotFoundError(f"Workbook not found: {OUTPUT_FILENAME}")

wb = openpyxl.load_workbook(OUTPUT_FILENAME)
for name in OLD_SHEET_NAMES:
    if name in wb.sheetnames:
        del wb[name]

source_header = wb["PDB_Data"]["A1"]

def format_sheet(ws, columns):
    for j, name in enumerate(columns, 1):
        c = ws.cell(1, j)
        c.font = copy(source_header.font); c.fill = copy(source_header.fill)
        c.border = copy(source_header.border)
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in ws.iter_rows():
        for c in row:
            if c.row > 1:
                c.alignment = Alignment(vertical="top", wrap_text=True)
            c.border = copy(source_header.border)
    ws.row_dimensions[1].height = 28
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False
    for cells in ws.columns:
        longest = max(len(str(c.value)) if c.value is not None else 0 for c in cells)
        ws.column_dimensions[cells[0].column_letter].width = min(max(longest + 2, 12), 45)

# ---------------- Ranges sheet ----------------
ws_r = wb.create_sheet(RANGES_SHEET)
range_headers = ["Parameter", "Minimum", "Maximum", "Median"]
ws_r.append(range_headers)
for rec in range_records:
    r = ws_r.max_row + 1
    ws_r.cell(r, 1, rec["Parameter"])
    s = rec["stats"]
    if s is None:
        for j in (2, 3, 4):
            ws_r.cell(r, j, NA_TEXT)
        continue
    put_number(ws_r.cell(r, 2), s["min"],    s["min_places"])
    put_number(ws_r.cell(r, 3), s["max"],    s["max_places"])
    put_number(ws_r.cell(r, 4), s["median_shown"], s["median_places"])
format_sheet(ws_r, range_headers)

# ---------------- One scoring sheet per hypothesis ----------------
col_idx = {name: i + 1 for i, name in enumerate(OUTPUT_COLUMNS)}
link_font = copy(wb["PDB_Data"].cell(2, 3).font)

def write_scoring_sheet(sheet_name, scoring_output):
    ws_s = wb.create_sheet(sheet_name)
    ws_s.append(OUTPUT_COLUMNS)
    for rec in scoring_output.to_dict("records"):
        r = ws_s.max_row + 1
        for name in OUTPUT_COLUMNS:
            cell = ws_s.cell(r, col_idx[name])
            val = rec[name]
            if name in SCORING_COLUMNS:                  # copy of PDB_Data value, exact decimals
                d = to_dec(val)
                if d is None:
                    cell.value = NA_TEXT
                else:
                    put_number(cell, d)
            elif name in SCORE_COLS:                     # calculated scores
                if pd.isna(val):
                    cell.value = NA_TEXT
                else:
                    cell.value = float(val); cell.number_format = "0.00"
            elif name == "Validation_Quality":
                if pd.isna(val):
                    cell.value = NA_TEXT
                else:
                    cell.value = int(val); cell.number_format = "0"
            elif name == "Rank":
                cell.value = int(val)
            else:                                        # text columns
                cell.value = NA_TEXT if (val is None or (isinstance(val, float) and pd.isna(val))
                                         or str(val).strip() == "") else val
    format_sheet(ws_s, OUTPUT_COLUMNS)
    for r in range(2, ws_s.max_row + 1):
        cell = ws_s.cell(r, col_idx["PDB Link"])
        pid = str(cell.value).strip()
        if pid and pid != NA_TEXT:
            cell.hyperlink = f"https://www.rcsb.org/structure/{pid}"
            cell.font = copy(link_font)

for sheet_name, out in scoring_outputs.items():
    write_scoring_sheet(sheet_name, out)

wb.save(OUTPUT_FILENAME)

# ---------------- Final summary ----------------
print("=" * 60)
print("PDB QUALITY SCORING COMPLETED")
print("=" * 60)
print("Output file :", OUTPUT_FILENAME)
print("PDB records :", len(score_df))
print("Added sheets:", RANGES_SHEET, "|", " | ".join(HYPOTHESES))
for sheet_name, out in scoring_outputs.items():
    if out.empty:
        print(f"{sheet_name}: no PDB structures available for scoring.")
        continue
    best = out.iloc[0]
    w = HYPOTHESES[sheet_name]
    print()
    print(f"{sheet_name}  weights: " + ", ".join(f"{k.split('_')[0]}={v:.2f}" for k, v in w.items()))
    print(f"  Best PDB: {best['PDB ID']}  | Final Score: {best['Final_Score']}  | Rank: {best['Rank']}")
print("=" * 60)



PDB QUALITY SCORING COMPLETED
Output file : CDK4_PDB_AlphaFold_Data.xlsx
PDB records : 11
Added sheets: Ranges | PDB_Scoring_H1 | PDB_Scoring_H2 | PDB_Scoring_H3

PDB_Scoring_H1  weights: Resolution=0.25, Rdiff=0.25, Residue=0.10, Validation=0.20, Bfactor=0.20
  Best PDB: 9CSK  | Final Score: 0.8  | Rank: 1

PDB_Scoring_H2  weights: Resolution=0.20, Rdiff=0.20, Residue=0.10, Validation=0.25, Bfactor=0.25
  Best PDB: 9CSK  | Final Score: 0.8  | Rank: 1

PDB_Scoring_H3  weights: Resolution=0.20, Rdiff=0.20, Residue=0.20, Validation=0.20, Bfactor=0.20
  Best PDB: 6P8E  | Final Score: 0.8  | Rank: 1
